In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if api_key and api_key.startswith('gsk_') and len(api_key)>10:
  print("API key looks good so far")
else:
  print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

MODEL = 'openai/gpt-oss-20b'
openai = OpenAI(
  api_key=os.getenv("GROQ_API_KEY"),
  base_url="https://api.groq.com/openai/v1"
)

API key looks good so far


In [5]:
links = fetch_website_links("https://anthropic.com")
links

['#main',
 '#footer',
 'https://www.anthropic.com/',
 'https://www.anthropic.com/research',
 'https://www.anthropic.com/research/team/alignment',
 'https://www.anthropic.com/research/team/economics',
 'https://www.anthropic.com/engineering',
 'https://www.anthropic.com/research/team/frontier-red-team',
 'https://www.anthropic.com/research/team/interpretability',
 'https://www.anthropic.com/science',
 'https://www.anthropic.com/research/team/societal-impacts',
 'https://www.anthropic.com/policy',
 'https://www.anthropic.com/constitution',
 'https://www.anthropic.com/claude-corps',
 'https://www.anthropic.com/policy-on-the-ai-exponential',
 'https://www.anthropic.com/transparency',
 'https://www.anthropic.com/responsible-scaling-policy',
 'https://www.anthropic.com/beneficial-deployments',
 'http://trust.anthropic.com/',
 'https://academy.claude.com',
 'https://claude.com/resources/tutorials',
 'https://claude.com/resources/use-cases',
 'https://platform.claude.com/docs',
 'https://www.a

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://anthropic.com"))


Here is the list of links on the website https://anthropic.com -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#main
#footer
https://www.anthropic.com/
https://www.anthropic.com/research
https://www.anthropic.com/research/team/alignment
https://www.anthropic.com/research/team/economics
https://www.anthropic.com/engineering
https://www.anthropic.com/research/team/frontier-red-team
https://www.anthropic.com/research/team/interpretability
https://www.anthropic.com/science
https://www.anthropic.com/research/team/societal-impacts
https://www.anthropic.com/policy
https://www.anthropic.com/constitution
https://www.anthropic.com/claude-corps
https://www.anthropic.com/policy-on-the-ai-exponential
https://www.anthropic.com/transparency
https://www.anthropic.com/responsible-scaling-policy
https://www.anthropic.com/b

In [9]:
def get_relevant_links(url):
  response = openai.chat.completions.create(
    model=MODEL,
    messages=[
      {"role": "system", "content": link_system_prompt},
      {"role": "user", "content": get_links_user_prompt(url)}
    ],
    response_format={"type": "json_object"}
  )
  result = response.choices[0].message.content
  links = json.loads(result)
  return links

In [10]:
get_relevant_links("https://anthropic.com")

{'links': [{'type': 'about page', 'url': 'https://www.anthropic.com/'},
  {'type': 'company page', 'url': 'https://www.anthropic.com/company'},
  {'type': 'leadership page',
   'url': 'https://www.anthropic.com/company/leadership'},
  {'type': 'careers page', 'url': 'https://www.anthropic.com/careers'}]}